In [1]:
import os
import pandas as pd
from preprocessing_mask_segmentation import preprocessing as pre
from shape_features import shape_features as sf
from split import split
from resize_image import resize as res

In [2]:
# Constant variables
FOLDER_PATH = "../images/"
CSV_PATH = "../labels.csv"
MERGED_CSV_AND_FEATURE_PATH = "../features.csv"

In [3]:
import torch
import torch.nn as nn
import numpy as np
import cv2

class SimpleCNNExtractor(nn.Module):
    def __init__(self, input_size=(128, 128)):
        super().__init__()
        self.input_size = input_size

        # Prosta architektura CNN (Feature Extractor)
        # Wejście: 1 kanał (maska binarna)
        self.features = nn.Sequential(
            # Blok 1
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # 128 -> 64

            # Blok 2
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # 64 -> 32

            # Blok 3
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            # Global Average Pooling - kluczowe dla niezmienniczości względem przesunięć
            # Spłaszcza obraz 32x32x64 do wektora 1x64 (średnia z każdej mapy cech)
            nn.AdaptiveAvgPool2d((1, 1))
        )

        # Inicjalizacja wag (opcjonalna, PyTorch robi to domyślnie dobrze)
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')

    def forward(self, img_array):
        # Resize
        # Wewnątrz pętli lub w klasie extractora przed resize:
        dist_transform = cv2.distanceTransform(img.astype('uint8'), cv2.DIST_L2, 5)
        # Normalizacja mapy odległości do 0-1, żeby CNN to zrozumiało
        if dist_transform.max() > 0:
            dist_transform = dist_transform / dist_transform.max()
        img_resized = cv2.resize(img_array.astype('float32'), self.input_size)

        # --- POPRAWKA: NORMALIZACJA ---
        # Jeśli max wartość to > 1 (np. 255), podziel przez 255.
        if img_resized.max() > 1.0:
            img_resized /= 255.0
        # ------------------------------

        tensor = torch.from_numpy(img_resized).unsqueeze(0).unsqueeze(0)

        # 2. Ekstrakcja
        self.eval() # Tryb ewaluacji (wyłącza dropout itp.)
        with torch.no_grad():
            output = self.features(tensor) # Wynik: [1, 64, 1, 1]
            feature_vector = output.flatten().numpy()

        # 3. Pakowanie do słownika
        features_dict = {}
        for i, val in enumerate(feature_vector):
            features_dict[f'CNN_Feat_{i}'] = val

        return features_dict

# Instancja ekstraktora (stwórz raz poza pętlą)
cnn_extractor = SimpleCNNExtractor()

In [4]:
images_names = sorted([f for f in os.listdir(FOLDER_PATH) if f.endswith('.png')])

# Import and preprocess images
images_dict = {}
for image_name in images_names:
    full_path = os.path.join(FOLDER_PATH, image_name)
    key = image_name.split('.')[0]
    img = pre.clean_mask(full_path)
    img = pre.smooth_mask_edges(img)

    # Image Resizing - uncomment if CNN used
    # img_resized = res.resize_with_padding(img)
    images_dict[key] = img

# Extract features from images
features_results = []

for name, img in images_dict.items():
    features = sf.extract_shape_features(img)
    # Jeśli maska jest pusta, pomiń
    if features is None:
        continue

    # 2. NOWE cechy (CNN)
    try:
        cnn_features = cnn_extractor(img)
        features.update(cnn_features) # Łączymy słowniki
    except Exception as e:
        print(f"Błąd CNN dla {name}: {e}")

    features['id'] = name
    features_results.append(features)


# Zamiana listy słowników na DataFrame
df_features = pd.DataFrame(features_results)

# Wczytanie etykiet (zakładam, że masz plik z mapowaniem id -> pathology)
# Jeśli labels_df to twoje oryginalne dane:
labels_df = pd.read_csv(CSV_PATH) # np. features.csv lub mass_case_description


# 1. Wymuś ten sam typ danych (string) dla klucza łączenia w obu tabelach
df_features['id'] = df_features['id'].astype(str)
labels_df['id'] = labels_df['id'].astype(str)

# 2. Opcjonalnie: usuń ewentualne spacje, które mogą psuć łączenie
df_features['id'] = df_features['id'].str.strip()
labels_df['id'] = labels_df['id'].str.strip()

# 3. Teraz Twoje łączenie zadziała
df_final = pd.merge(df_features, labels_df[['id', 'pathology']], on='id', how='inner')

# DIAGNOSTYKA: Sprawdź, czy łączenie nie wyszło puste!
print(f"Liczba próbek przed łączeniem: {len(df_features)}")
print(f"Liczba próbek po łączeniu: {len(df_final)}")

if len(df_final) == 0:
    print("ALARM: Łączenie zwróciło pusty zbiór! Sprawdź, czy wartości ID w obu tabelach są identyczne (np. '1' vs '001').")
    print(f"Przykładowe ID z cech: {df_features['id'].head().tolist()}")
    print(f"Przykładowe ID z etykiet: {labels_df['id'].head().tolist()}")

# ŁĄCZENIE (MERGE) - to jest jedyny bezpieczny sposób
# 'inner' usunie te wiersze, dla których nie udało się wygenerować cech
df_final = pd.merge(df_features, labels_df[['id', 'pathology']], on='id', how='inner')

# Dopiero teraz definiujesz X i y
X = df_final.drop(['pathology', 'id', 'patient_id'], axis=1, errors='ignore')
y = df_final['pathology']


Liczba próbek przed łączeniem: 1664
Liczba próbek po łączeniu: 1664


In [5]:
# Import CSV file and
csv_file = pd.read_csv(CSV_PATH)

# Convert features_results into Pandas DataFrame
df_feature_results = pd.DataFrame(features_results)

# Convert id columns to string to prepare merge
csv_file['id'] = csv_file['id'].astype(str)
df_feature_results["id"] = df_feature_results["id"].astype(str)

# Merge CSV columns and feature DataFrame into one final DataFrame
df_final = pd.merge(csv_file, df_feature_results, left_on='id', right_on='id')

# Drop useless columns and convert target feature into binary value
df_final.drop(columns='abnormality_type', inplace=True)
# df_final['pathology'] = (1 if df_final['pathology'] == "MALIGNANT" else 0)
mapping = {'MALIGNANT': 1, 'BENIGN': 0}
df_final['pathology'] = df_final['pathology'].map(mapping)

# Export final DataFrame into CSV file
df_final.to_csv(MERGED_CSV_AND_FEATURE_PATH, index=False)

## Test of Feature Extraction and Preprocessing

In [6]:
df_final.head()

,id,patient_id,assessment,pathology,Area,Area Bounding Box,Area Convex,Area Filled,Axis Major Length,Axis Minor Length,...,CNN_Feat_54,CNN_Feat_55,CNN_Feat_56,CNN_Feat_57,CNN_Feat_58,CNN_Feat_59,CNN_Feat_60,CNN_Feat_61,CNN_Feat_62,CNN_Feat_63
0,1001,P_00001,4,1,120176.0,184775.0,144592.0,120176.0,437.817532,369.842129,...,0.000125,0.058044,0.013867,0.002505,0.000350,0.116401,0.061270,0.121181,0.0,0.002918
1,1002,P_00001,4,1,32114.0,54432.0,39160.0,32114.0,215.156997,204.049262,...,0.000039,0.054782,0.014940,0.002742,0.000379,0.108820,0.056379,0.115270,0.0,0.002817
2,1003,P_00004,4,0,113057.0,162306.0,132259.0,113057.0,403.087349,373.343420,...,0.000044,0.064115,0.013213,0.003035,0.000306,0.127986,0.068926,0.127684,0.0,0.002674
3,1004,P_00004,4,0,89182.0,147452.0,111392.0,89182.0,361.346738,333.637192,...,0.000045,0.054829,0.014751,0.002434,0.000408,0.113181,0.058446,0.120541,0.0,0.002891
4,1005,P_00004,4,0,91546.0,151434.0,108858.0,91546.0,389.640992,311.266317,...,0.000052,0.056616,0.013871,0.003005,0.000301,0.111862,0.057688,0.117540,0.0,0.002722


In [7]:
df_final.describe()

,assessment,pathology,Area,Area Bounding Box,Area Convex,Area Filled,Axis Major Length,Axis Minor Length,Centroid X,Centroid Y,...,CNN_Feat_54,CNN_Feat_55,CNN_Feat_56,CNN_Feat_57,CNN_Feat_58,CNN_Feat_59,CNN_Feat_60,CNN_Feat_61,CNN_Feat_62,CNN_Feat_63
count,1664.000000,1664.000000,1.664000e+03,1.664000e+03,1.664000e+03,1.664000e+03,1664.000000,1664.000000,1664.000000,1664.000000,...,1664.000000,1664.000000,1664.000000,1664.000000,1664.000000,1664.000000,1664.000000,1664.000000,1664.0,1664.000000
mean,3.501202,0.458534,7.868357e+04,1.221009e+05,9.073593e+04,7.868357e+04,334.169193,269.915583,206.031864,202.283244,...,0.000033,0.056874,0.014285,0.002618,0.000320,0.114425,0.060992,0.118063,0.0,0.002621
std,1.404828,0.498427,8.055054e+04,1.206283e+05,9.094821e+04,8.055054e+04,137.835099,113.287892,83.124890,81.275151,...,0.000029,0.005440,0.001047,0.000240,0.000025,0.009102,0.006858,0.004498,0.0,0.000336
min,0.000000,0.000000,3.966000e+03,6.095000e+03,4.643000e+03,3.966000e+03,80.596272,52.545921,51.516357,53.855270,...,0.000000,0.034286,0.009253,0.001969,0.000279,0.080082,0.037487,0.098358,0.0,0.001658
25%,3.000000,0.000000,3.621375e+04,5.792400e+04,4.223800e+04,3.621375e+04,245.383208,195.395222,152.343274,148.652792,...,0.000010,0.053438,0.013614,0.002451,0.000301,0.108760,0.056504,0.115383,0.0,0.002386
50%,4.000000,0.000000,5.651000e+04,8.885800e+04,6.547500e+04,5.651000e+04,305.682136,247.529924,188.373131,186.009010,...,0.000028,0.056966,0.014208,0.002611,0.000311,0.114715,0.061028,0.118404,0.0,0.002595
75%,4.000000,1.000000,9.042775e+04,1.425425e+05,1.045720e+05,9.042775e+04,383.560436,318.783599,237.281471,236.437932,...,0.000049,0.060370,0.014944,0.002761,0.000328,0.120300,0.065490,0.120920,0.0,0.002825
max,5.000000,1.000000,1.251936e+06,1.797881e+06,1.345671e+06,1.251936e+06,1317.697863,1222.367729,811.905826,791.568087,...,0.000222,0.082418,0.019165,0.003809,0.000516,0.158958,0.094526,0.135852,0.0,0.004019


## Data Split
Perform `patient_id` aware split. Extract X features and y labels into separate DataFrames. Split X and y into train and test sets.
## (NOTE: CHANGE DROP_COLS LIST WHEN ALL NEEDED FEATURES ARE SELECTED)

In [8]:
from sklearn.preprocessing import StandardScaler


drop_cols = ['id', 'patient_id', 'pathology']
X_train, X_test, y_train, y_test, is_intersect_empty = split.aware_patient_split(df_final, drop_cols)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Verify that the intersection of patient IDs in Train and Test sets is empty
is_intersect_empty # Result: True

True

## Model Training

Basic model training - for test purposes.

In [9]:
from models import models

lr = models.train_lr(X_train, y_train)
svm = models.train_svm(X_train, y_train)
rf = models.train_rf(X_train, y_train)
knn = models.train_knn(X_train, y_train)

result_lr = models.test_model(lr, X_test, y_test)
result_svm = models.test_model(svm, X_test, y_test)
result_rf = models.test_model(rf, X_test, y_test)
result_knn = models.test_model(knn, X_test, y_test)

C:\Users\KONRAD_ADMIN\Desktop\mammography-delta\.venv\lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## Model Results

Basic model results - for testing purposes

## (NOTE: BECAUSE OF FEATURES CORRELATION AND NO FEATURE SCALLING, RESULTS ARE RANDOM RIGHT NOW)

In [10]:
print(result_lr)

[[133  51]
 [ 54 103]]


In [11]:
print(result_svm)

[[138  46]
 [ 54 103]]


In [12]:
print(result_rf)

[[142  42]
 [ 59  98]]


In [13]:
print(result_knn)

[[112  72]
 [ 76  81]]
